In [1]:
import json, math
from datetime import datetime
from pathlib import Path
from sentence_transformers import SentenceTransformer

NODES = Path("nodes.json")
EMBS  = Path("embeddings.json")

MODEL = "all-MiniLM-L6-v2"
TOP_K = 5

# paper-ish knobs
HALF_LIFE_HOURS = 24.0
REL_MIN = 0.62  # keep this unless you want more/less filtering

TYPE_W = {"thought": 2.0, "chat": 1.7, "event": 1.0}

def parse_dt(s):
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M"):
        try: return datetime.strptime(str(s), fmt)
        except: pass
    return datetime.fromisoformat(str(s))

def cosine(a, b):
    if a is None or b is None or len(a) != len(b): return 0.0
    dot = sum(x*y for x, y in zip(a, b))
    na = math.sqrt(sum(x*x for x in a))
    nb = math.sqrt(sum(y*y for y in b))
    return dot/(na*nb) if na and nb else 0.0

def recency(created, now, half_life_hours):
    dt_h = max(0.0, (now - created).total_seconds()/3600.0)
    return 2 ** (-dt_h / half_life_hours)

def importance(poignancy):
    p = float(poignancy or 0.0)
    if p <= 1.0:
        return p / 10.0
    return min(1.0, p / 10.0)

def load_nodes():
    data = json.loads(NODES.read_text(encoding="utf-8"))
    nodes = list(data.values()) if isinstance(data, dict) else data
    for n in nodes:
        n["_t"] = parse_dt(n["created"])
        n["_k"] = (n.get("embedding_key") or n.get("description") or n.get("text") or "").strip()
    nodes.sort(key=lambda x: x["_t"])
    return nodes

def load_embs():
    e = json.loads(EMBS.read_text(encoding="utf-8"))
    return {k: v for k, v in e.items() if isinstance(v, list)}

def retrieve(nodes, embs, qvec, *,
             w_recency, w_importance, w_relevance,
             now=None, top_k=TOP_K, rel_min=REL_MIN, half_life_hours=HALF_LIFE_HOURS):
    now = now or nodes[-1]["_t"]
    out = []
    for n in nodes:
        # optional expiration handling
        if n.get("expiration"):
            try:
                if parse_dt(n["expiration"]) <= now:
                    continue
            except:
                pass

        vec = embs.get(n["_k"])
        if vec is None:
            continue

        rel = (cosine(qvec, vec) + 1) / 2  # [-1,1]→[0,1]
        if rel < rel_min:
            continue

        r = recency(n["_t"], now, half_life_hours)
        imp = importance(n.get("poignancy"))
        tw = TYPE_W.get(n.get("type"), 1.0)

        score = tw * (w_recency*r + w_importance*imp + w_relevance*rel)
        out.append((score, n))

    out.sort(key=lambda x: x[0], reverse=True)
    return out[:top_k]

def fmt_desc(n, n_chars=140):
    return (n.get("description") or n.get("text") or "").replace("\n", " ")[:n_chars]

if __name__ == "__main__":
    nodes = load_nodes()
    embs = load_embs()

    model = SentenceTransformer(MODEL)

    # ✅ Use your own Q4/Q5 (I put examples; replace them)
    experiments = [
        {
            "query": "Valentine's Day party balloons snacks decorations",
            "w": (0.05, 0.30, 3.20),   # relevance-heavy (find “party supplies” memories)
        },
        {
            "query": "Who did Klaus talk to about the Valentine's Day party?",
            "w": (0.20, 0.30, 3.00),   # still relevance-heavy + a bit more recency
        },
        {
            "query": "Who are Klaus's close friends?",
            "w": (0.10, 1.20, 2.00),   # importance-heavy (relationship memories often have poignancy)
        },
        {
            "query": "What is Klaus working?",  # Q4 (example)
            "w": (0.15, 0.60, 2.50),   # balanced-ish
        },
        {
            "query": "What did Klaus do earlier today?",            # Q5 (example)
            "w": (0.90, 0.10, 1.80),   # recency-heavy (surface latest timeline)
        },
    ]

    for ex in experiments:
        query = ex["query"]
        w_recency, w_importance, w_relevance = ex["w"]

        qvec = model.encode([query], normalize_embeddings=False)[0].tolist()
        top = retrieve(nodes, embs, qvec,
                       w_recency=w_recency, w_importance=w_importance, w_relevance=w_relevance)

        # ---- Assignment-friendly print block ----
        print("```python")
        print(f'Query: "{query}"')
        print(f"Weights: W_RECENCY={w_recency}, W_IMPORTANCE={w_importance}, W_RELEVANCE={w_relevance}")
        print("\nOutput:")
        for score, n in top:
            print(f'{score:.3f} | {n.get("type")} | {n.get("created")} | {fmt_desc(n)}')
        print("```")
        print()

c:\Users\srina\projects\Generative Agents and Memory-Based Retrieval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 758.03it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


```python
Query: "Valentine's Day party balloons snacks decorations"
Weights: W_RECENCY=0.05, W_IMPORTANCE=0.3, W_RELEVANCE=3.2

Output:
5.339 | thought | 2023-02-13 17:34:50 | For Klaus Mueller's planning: needs to remember to bring some red and pink balloons and homemade cookies and chips for the Valentine's Day p
4.861 | thought | 2023-02-13 11:24:50 | Klaus Mueller Klaus found the invitation to the Valentine's Day party interesting.
4.770 | thought | 2023-02-13 11:24:50 | For Klaus Mueller's planning: needs to remember to attend Isabella Rodriguez's Valentine's Day party at Hobbs Cafe on February 14th, 2023 fr
4.667 | thought | 2023-02-13 17:34:50 | Klaus Mueller Klaus Mueller found it interesting that Isabella asked for his help with the decorations and snacks for the party.
4.638 | thought | 2023-02-13 17:47:20 | Klaus Mueller is invited to Isabella Rodriguez's Valentine's Day party at Hobbs Cafe
```

```python
Query: "Who did Klaus talk to about the Valentine's Day party?"
Weigh